# Full Preprocessing Pipeline (Verbose & Multi-Task)

This notebook implements the complete preprocessing strategy defined in `PREPROCESSING_GUIDE.md` with **live logging**.

## Objectives:
1. **Initial Validation**: Shuffle, drop duplicates and nulls from the raw dataset.
2. **Structural Cleaning**: Remove GIFs, stickers, and tags. Drop tag-only comments.
3. **Text Normalization**: Map punctuation intensity to `[INTENSE]`.
4. **Multi-Task Emoji Mapping**: Map emojis to tokens based on the selected task.
5. **Language Filtering**: Keep only Arabic and Latin scripts.
6. **Full Dataset Export**: Save the entire cleaned dataset with **Random IDs**.

In [9]:
import pandas as pd
import re
import emoji
import random
import numpy as np
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 42
INPUT_FILE = 'dataset.csv'
SAMPLE_SIZE = 1000 # Adjust for verbose speed

## 1. Initial Data Cleaning
Before we select a task, we perform a global shuffle, deduplication, and null-check.

In [10]:
print(f"Loading {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)
initial_count = len(df)

# 1. Drop absolute empty comments
df = df.dropna(subset=['comment_text'])
after_nulls = len(df)

# 2. Shuffle globally
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# 3. Drop duplicates in comment_text
df = df.drop_duplicates(subset=['comment_text'])
after_dedup = len(df)

print(f"--- Initial Cleaning Results ---")
print(f"Raw rows: {initial_count}")
print(f"Dropped {initial_count - after_nulls} null rows.")
print(f"Global shuffle applied.")
print(f"Dropped {after_nulls - after_dedup} duplicate rows.")
print(f"Total unique usable rows: {len(df)}")

Loading dataset.csv...
--- Initial Cleaning Results ---
Raw rows: 43788
Dropped 82 null rows.
Global shuffle applied.
Dropped 9252 duplicate rows.
Total unique usable rows: 34454


## 2. Select Preprocessing Task

In [11]:
print("Select Preprocessing Task:")
print("1. Sentiment Analysis")
print("2. Intent Classification")
print("3. Topic Classification")

choice = input("Enter choice (1/2/3): ")
modes = {"1": "SENTIMENT", "2": "INTENT", "3": "TOPIC"}
MODE = modes.get(choice, "SENTIMENT")
print(f"\n>>> Active Mode: {MODE}")

Select Preprocessing Task:
1. Sentiment Analysis
2. Intent Classification
3. Topic Classification

>>> Active Mode: INTENT


## 3. Core Functions

In [14]:
def log_transform(step_name, original, result):
    if str(original).strip() != str(result).strip():
        print(f"  [{step_name}] ")
        print(f"    - Before: {original}")
        print(f"    - After : {result}")

def full_pipeline(text, mode, verbose=True):
    if not isinstance(text, str): return ""
    current = text.strip()
    if verbose: print(f"\n--- Processing: '{current}'")

    # 1. Structural Cleaning (GIFs, Stickers, Replying to prefixes)
    temp = re.sub(r"\[GIF\]|\[Sticker\]", "", current)
    temp = re.sub(r"Replying to @[\w.]+[: ]*", "", temp)
    if verbose: log_transform("Structural", current, temp)
    current = temp

    # 2. Advanced Tag & Flair Removal
    temp = re.sub(r"(@[\w.]+(?:[^\w\s\u0600-\u06FF])*)", "", current)
    temp = temp.strip()
    if not temp:
        if verbose: print("  [DROP] Reason: Tags only comment")
        return ""
    if verbose: log_transform("Tag & Flair Removal", current, temp)
    current = temp

    # 3. Punctuation Intensity
    temp = re.sub(r"[!]{2,}|[?]{2,}|[.]{3,}", " [INTENSE] ", current).strip()
    if verbose: log_transform("Intensity", current, temp)
    current = temp

    # 4. Emoji Mapping
    emojis_found = [e['emoji'] for e in emoji.emoji_list(current)]
    tokens = []
    
    if mode == "SENTIMENT":
        pos = ['❤️', '🥰', '😍', '🔥', '😋', '😂', '👏', '💯', '👍', '😁', '🤩', '😊', '🥳', '💪', '🤲', '🌹', '💐', '💎', '🇩🇿']
        neg = ['🤮', '😡', '👎', '💔', '💀', '💸', '😭', '😢', '😒', '😑', '😱']
        if any(e in pos for e in emojis_found): tokens.append("[POS]")
        if any(e in neg for e in emojis_found): tokens.append("[NEG]")

    elif mode == "INTENT":
        appr = ['❤️', '🥰', '😂', '👏', '🤲', '🌹', '💐']
        comp = ['🤮', '😡', '👎', '💔', '💀', '💸', '😒', '😑']
        inq = ['❓', '❔', '🤔', '🧐', '👀', '📍', '📞', '🕒']
        recom = ['👌', '🔝', '🌟', '✨', '✅', '🥇', '👑']
        if any(e in appr for e in emojis_found): tokens.append("[APPRECIATION]")
        if any(e in comp for e in emojis_found): tokens.append("[COMPLAINT]")
        if any(e in inq for e in emojis_found): tokens.append("[INQUIRY]")
        if any(e in recom for e in emojis_found): tokens.append("[RECOMMENDATION]")
        # if not tokens and emojis_found: tokens.append("[OUT_OF_SCOPE]")

    elif mode == "TOPIC":
        bouffe = ['🥘', '🍔', '🍕', '🥙', '🥗', '🍦', '😋', '🤤', '🍜', '🍣', '🥩']
        price = ['💸', '💰', '💳', '💶', '💵']
        treat = ['🧑‍🍳', '👨‍🍳', '👋', '🤝', '🫂']
        srv = ['🕒', '⏳', '🛵', '🍴', '🍽️']
        endroit = ['📍', '🧼', '🧹', '📸', '🤳', '✨', '🌟', '🏝']
        delivery = ['🛵', '🚚', '📦']
        if any(e in bouffe for e in emojis_found): tokens.append("[BOUFFE]")
        if any(e in price for e in emojis_found): tokens.append("[PRICE]")
        if any(e in treat for e in emojis_found): tokens.append("[TREATMENT]")
        if any(e in srv for e in emojis_found): tokens.append("[SERVICE]")
        if any(e in endroit for e in emojis_found): tokens.append("[ENDROIT]")
        if any(e in delivery for e in emojis_found): tokens.append("[DELIVERY]")
        # if not tokens and emojis_found: tokens.append("[UNKNOWN]")

    temp = emoji.replace_emoji(current, replace="")
    final = (temp + " " + " ".join(tokens)).strip()
    if verbose: log_transform("Emoji Mapping", current, final)
    
    return final

def is_target_language(text):
    if len(text.strip()) < 2: return False
    try:
        lang = detect(text)
        return lang in ['ar', 'fr', 'en']
    except:
        return False

def generate_random_ids(n):
    """Generates unique random 6-digit IDs"""
    ids = set()
    while len(ids) < n:
        ids.add(random.randint(100000, 999999))
    return list(ids)

## 4. Test on a Sample (Verbose)

In [15]:
sample_df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)
sample_df['raw_text'] = sample_df['comment_text']

print(f"Executing Pipeline (Sample) for {MODE} mode...\n")
sample_df['comment_text'] = sample_df['comment_text'].apply(lambda x: full_pipeline(x, MODE, verbose=True))

pd.set_option('display.max_colwidth', None)
display(sample_df[['raw_text', 'comment_text']].head(10))

Executing Pipeline (Sample) for INTENT mode...


--- Processing: 'la kabylie d’hier et d’aujourd’hui , une nostalgie du civisme et de la propreté perdu.....on dirait Kaboul.'
  [Intensity] 
    - Before: la kabylie d’hier et d’aujourd’hui , une nostalgie du civisme et de la propreté perdu.....on dirait Kaboul.
    - After : la kabylie d’hier et d’aujourd’hui , une nostalgie du civisme et de la propreté perdu [INTENSE] on dirait Kaboul.

--- Processing: 'c'est des prix très raisonnables .'

--- Processing: '@Houdaa Yadra kima gotlek'
  [Tag & Flair Removal] 
    - Before: @Houdaa Yadra kima gotlek
    - After : Yadra kima gotlek

--- Processing: 'ديكور برك و ضيق بزاف تقول راك قاعد معاهم كامل والماكلة على ربي و les prix منحكوش'

--- Processing: 'je suis kabyle 🥰'
  [Emoji Mapping] 
    - Before: je suis kabyle 🥰
    - After : je suis kabyle  [APPRECIATION]

--- Processing: 'It's so good but u have to clean more especially the couch they are dirty honestly'

--- Processing: 'كاين في باتنة

,raw_text,comment_text
0,"la kabylie d’hier et d’aujourd’hui , une nostalgie du civisme et de la propreté perdu.....on dirait Kaboul.","la kabylie d’hier et d’aujourd’hui , une nostalgie du civisme et de la propreté perdu [INTENSE] on dirait Kaboul."
1,c'est des prix très raisonnables .,c'est des prix très raisonnables .
2,@Houdaa Yadra kima gotlek,Yadra kima gotlek
3,ديكور برك و ضيق بزاف تقول راك قاعد معاهم كامل والماكلة على ربي و les prix منحكوش,ديكور برك و ضيق بزاف تقول راك قاعد معاهم كامل والماكلة على ربي و les prix منحكوش
4,je suis kabyle 🥰,je suis kabyle [APPRECIATION]
5,It's so good but u have to clean more especially the couch they are dirty honestly,It's so good but u have to clean more especially the couch they are dirty honestly
6,كاين في باتنة,كاين في باتنة
7,F alger,F alger
8,"A notre habitude nos gateaux d'annivarisaire sont tjr commandés chez vous mais ces detniers temps bcp de faux pas! Hormis le fait que le gateau d'annivraissaire ne ressemblait pas dutt a ce que j'ai demandé ou ce qui est pris en photo sur leurs reseaux, en ayant fait la remarque a la vendense sa reponse etait (Mme koul nhar ou nharo d' apres elle)\nVos gateaux etaient plus fins,plus precis, decorés avec delicatesse, la c'etait une deco digne d' un gateau fait maison, j'ai du le prendre car c' etait en fin de journèe et pas d' autre choix!\nPs: la politesse fait partie de votre job!","A notre habitude nos gateaux d'annivarisaire sont tjr commandés chez vous mais ces detniers temps bcp de faux pas! Hormis le fait que le gateau d'annivraissaire ne ressemblait pas dutt a ce que j'ai demandé ou ce qui est pris en photo sur leurs reseaux, en ayant fait la remarque a la vendense sa reponse etait (Mme koul nhar ou nharo d' apres elle)\nVos gateaux etaient plus fins,plus precis, decorés avec delicatesse, la c'etait une deco digne d' un gateau fait maison, j'ai du le prendre car c' etait en fin de journèe et pas d' autre choix!\nPs: la politesse fait partie de votre job!"
9,4h pour manger ! Fait trop chaud !,4h pour manger ! Fait trop chaud !


## 5. Process & Export Full Dataset

In [ ]:
recap = {}
recap['1_Load'] = len(df)

print(f"Applying pipeline to FULL dataset ({len(df)} rows)... Mode: {MODE}")
final_df = df.copy()

final_df['comment_text'] = final_df['comment_text'].apply(lambda x: full_pipeline(x, MODE, verbose=False))
recap['2_Cleaned'] = len(final_df)

print("Removing empty results...")
final_df = final_df[final_df['comment_text'].str.strip() != ""]
final_df = final_df.dropna(subset=['comment_text'])
recap['3_NonEmpty'] = len(final_df)

print("Enforcing script rules (Arabic/Latin)...")
final_df = final_df[final_df['comment_text'].apply(is_target_language)]
recap['4_Language'] = len(final_df)

print("Final deduplication and shuffle...")
final_df = final_df.drop_duplicates(subset=['comment_text'])
recap['5_FinalUnique'] = len(final_df)

print("Assigning Random IDs...")
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)
final_df.insert(0, 'final_id', generate_random_ids(len(final_df)))

output_file = f'dataset_{MODE.lower()}_preprocessed.csv'
final_df.to_csv(output_file, index=False)

print(f"\nSUCCESS! Saved {len(final_df)} high-quality rows to {output_file}")

Applying pipeline to FULL dataset (34454 rows)... Mode: INTENT
Removing empty results...
Enforcing script rules (Arabic/Latin)...
Final deduplication and shuffle...
Assigning Random IDs...

SUCCESS! Saved 24819 high-quality rows to dataset_intent_preprocessed.csv


## 6. Preprocessing Recap

In [17]:
recap_df = pd.DataFrame.from_dict(recap, orient='index', columns=['Row Count'])
recap_df['Yield %'] = (recap_df['Row Count'] / recap['1_Load'] * 100).round(2)
recap_df['Dropped'] = recap_df['Row Count'].diff().fillna(0).astype(int) * -1

print(f"\n--- Preprocessing Recap: {MODE} Mode ---")
display(recap_df)
print(f"\nTotal usable data preserved: {recap_df.iloc[-1]['Yield %']}% of unique raw input.")


--- Preprocessing Recap: INTENT Mode ---


,Row Count,Yield %,Dropped
1_Load,34454,100.00,0
2_Cleaned,34454,100.00,0
3_NonEmpty,33521,97.29,933
4_Language,25151,73.00,8370
5_FinalUnique,24819,72.04,332



Total usable data preserved: 72.04% of unique raw input.
